# Burgers equation with PhiJAX and Hydra

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HangJung97/phijax-hydra-template/blob/main/notebooks/burgers_example.ipynb)

This notebook runs the complete Burgers reference experiment locally or on Google Colab. It composes the Hydra config, trains the configured physics-informed neural network for 5,000 steps, streams compact training metrics, evaluates the saved prediction artifact, and plots the result.

From the project root, install and start the optional notebook environment with:

```bash
# CPU
uv sync --extra jupyter

# NVIDIA GPU: choose exactly one CUDA extra
uv sync --extra jupyter --extra cuda12
# or
uv sync --extra jupyter --extra cuda13

# Start Jupyter after installing the selected environment
uv run --no-sync jupyter lab notebooks/burgers_example.ipynb
```

## Problem

We solve the one-dimensional viscous Burgers equation

$$
\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} - \nu\frac{\partial^2 u}{\partial x^2} = 0,
\qquad (t,x) \in (0,1] \times [-1,1],
\qquad \nu = \frac{0.01}{\pi},
$$

subject to the initial condition

$$
u(0,x) = -\sin(\pi x), \qquad x \in [-1,1],
$$

and periodic boundary conditions with period 2:

$$
u(t,-1) = u(t,1), \qquad
\frac{\partial u}{\partial x}(t,-1) = \frac{\partial u}{\partial x}(t,1),
\qquad t \in [0,1].
$$

The model's periodic coordinate embedding enforces the boundary conditions without a separate boundary-loss term.

## Set up the demonstration

On Colab, the setup cell clones the project and creates its isolated `uv` environment. Select a GPU runtime and leave `USE_GPU = True` for CUDA 12, or set it to `False` for CPU execution. Local users should install the matching CUDA extra before selecting GPU execution. The project entrypoints set up the project root, and Hydra creates each run directory from the composed config. The notebook passes PhiJAX's documented deterministic XLA options to every child command before JAX starts.

> **Note:** The Rich progress bar may render updates on separate lines. This visual glitch does not affect training.

In [ ]:
import codecs
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from collections.abc import Sequence
from pathlib import Path

from IPython.display import Image
from IPython.display import display as show

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
USE_GPU = True
if IN_COLAB:
    repository = Path("/content/phijax-hydra-template")
    if not (repository / "pyproject.toml").is_file():
        git_executable = shutil.which("git")
        if git_executable is None:
            raise RuntimeError("The Colab runtime does not provide git.")
        subprocess.run(  # noqa: S603 - clone the fixed public template repository.
            [
                git_executable,
                "clone",
                "--depth",
                "1",
                "https://github.com/HangJung97/phijax-hydra-template.git",
                str(repository),
            ],
            check=True,
        )
    os.chdir(repository)
    uv_executable = shutil.which("uv")
    if uv_executable is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "uv>=0.12,<0.13"],
            check=True,
        )
        uv_executable = shutil.which("uv")
    if uv_executable is None:
        raise RuntimeError("The Colab setup could not install uv.")
    sync_command = [uv_executable, "sync", "--locked", "--no-dev"]
    if USE_GPU:
        sync_command.extend(["--extra", "cuda12"])
    subprocess.run(sync_command, check=True)  # noqa: S603 - arguments are fixed above.
    os.environ["PATH"] = f"{repository / '.venv' / 'bin'}{os.pathsep}{os.environ['PATH']}"

FORCE_CPU = not USE_GPU
ENVIRONMENT = os.environ.copy()
DETERMINISTIC_XLA_FLAGS = "--xla_gpu_exclude_nondeterministic_ops --xla_gpu_autotune_level=0"
ENVIRONMENT["XLA_FLAGS"] = DETERMINISTIC_XLA_FLAGS
if FORCE_CPU:
    ENVIRONMENT["JAX_PLATFORMS"] = "cpu"

EXPERIMENT_OVERRIDES = [
    "experiment=burgers_grad_norm_1d",
]


def run_command(arguments: Sequence[str]) -> subprocess.CompletedProcess[str]:
    """Run one project command and stream its combined output.

    Args:
        arguments: Command and arguments passed directly to the child process.

    Returns:
        Completed successful child process containing the combined output.
    """
    print(f"$ {shlex.join(arguments)}")
    process = subprocess.Popen(  # noqa: S603 - arguments are fixed project commands and explicit overrides.
        list(arguments),
        env=ENVIRONMENT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    if process.stdout is None:
        raise RuntimeError("The child process did not expose its output stream.")
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    output_chunks = []
    while raw_chunk := os.read(process.stdout.fileno(), 4096):
        chunk = decoder.decode(raw_chunk)
        output_chunks.append(chunk)
        print(chunk, end="", flush=True)
    final_chunk = decoder.decode(b"", final=True)
    if final_chunk:
        output_chunks.append(final_chunk)
        print(final_chunk, end="", flush=True)
    completed = subprocess.CompletedProcess(
        args=list(arguments),
        returncode=process.wait(),
        stdout="".join(output_chunks),
        stderr=None,
    )
    completed.check_returncode()
    return completed

## Inspect the composed config

Hydra resolves the complete reference experiment without starting training.

In [ ]:
run_command(["phijax-train", *EXPERIMENT_OVERRIDES, "--cfg", "job", "--resolve"]);

## Train the reference PINN

The Burgers reference dataset is created automatically at `data/burgers/burgers.mat` when it is not already available. This run uses the same experiment config, except that `callbacks.model_checkpoint.every_n_steps=null` disables periodic checkpoint saves. The final checkpoint is still saved.

In [ ]:
TRAINING = run_command(
    [
        "phijax-train",
        *EXPERIMENT_OVERRIDES,
        "callbacks.model_checkpoint.every_n_steps=null",
    ]
)

prediction_match = re.search(r"Predictions saved to <([^>]+)>", TRAINING.stdout)
if prediction_match is None:
    raise RuntimeError("Training completed without reporting a prediction artifact.")
PREDICTION_PATH = Path(prediction_match.group(1))
assert PREDICTION_PATH.is_file()
PREDICTION_PATH

## Predict again from the checkpoint

This section shows how to use `phijax-predict` with the stored checkpoint. Training already created the prediction artifact used below, so the cell prints the equivalent command instead of running prediction again. Hydra creates a separate prediction run directory when you run the command.

In [ ]:
CHECKPOINT_DIR = PREDICTION_PATH.parents[1] / "checkpoints"
assert CHECKPOINT_DIR.is_dir()
print(shlex.join(["phijax-predict", *EXPERIMENT_OVERRIDES, f"ckpt_path={CHECKPOINT_DIR}"]))

## Evaluate the prediction artifact

Evaluation compares the saved prediction with the reference solution. It computes the relative L2 error, root mean squared error, mean absolute error, and maximum absolute error. The 5,000-step experiment should provide reasonable accuracy, while additional training steps may improve the result further.

In [ ]:
run_command(
    [
        "phijax-evaluate",
        f"predictions={PREDICTION_PATH}",
        "extras.print_config=false",
    ]
)

METRICS_PATH = PREDICTION_PATH.parent / "results" / "metrics.json"
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
metrics

## Plot the reference, prediction, and error

The first two panels show the reference and predicted Burgers solutions over time and space. The third panel shows the pointwise absolute error, $|u_{\mathrm{pred}} - u_{\mathrm{ref}}|$.

In [ ]:
FIGURE_PATH = PREDICTION_PATH.with_suffix(".png")
PLOT_SCRIPT = next(
    parent / "scripts" / "plot_burgers_predictions.py"
    for parent in PREDICTION_PATH.parents
    if (parent / "pyproject.toml").is_file()
)
run_command(["python", str(PLOT_SCRIPT), str(PREDICTION_PATH), "--output", str(FIGURE_PATH)])
show(Image(filename=str(FIGURE_PATH)))

## Next steps

Run the complete reference experiment with deterministic GPU execution:

```bash
XLA_FLAGS="--xla_gpu_exclude_nondeterministic_ops --xla_gpu_autotune_level=0" \
  phijax-train experiment=burgers_grad_norm_1d
```

Edit the experiment config to keep a new reproducible setup, and use command-line overrides for temporary comparisons.